# **1. Importação das bibliotecas**

In [2]:
import numpy as np
import math
import re
import time
import zipfile
import tensorflow as tf
from tensorflow.keras import layers
import random
import tensorflow_datasets as tfds

# **2. Limpeza dos dados**

In [3]:
with open('/content/drive/MyDrive/pt-en/europarl-v7.pt-en.en', mode='r', encoding='utf-8') as f:
  europarl_en = f.read()
with open('/content/drive/MyDrive/pt-en/europarl-v7.pt-en.pt', mode='r', encoding='utf-8') as f:
  europarl_pt = f.read()

In [4]:
europarl_en[0:100]

'Resumption of the session\nI declare resumed the session of the European Parliament adjourned on Frid'

In [5]:
en = europarl_en.split('\n')
len(en)

1960408

In [ ]:
en

In [7]:
pt = europarl_pt.split('\n')
len(pt)

1960408

In [8]:
for _ in range(5):
  print('-------')
  i = random.randint(0, len(en) - 1)
  print(en[i])
  print(pt[i])

-------
Nevertheless, the Greens welcome this move by the European Parliament towards greater comprehensibility, which will enable consumers to choose freely between products that contain GMOs and those that do not.
Não obstante, os Verdes congratulam-se com este avanço do PE na direcção de uma maior legibilidade para o consumidor, para que este possa escolher livremente entre os produtos que contêm OGM e os que estão deles isentos.
-------
Drafting in child soldiers not only threatens to lead to a criminalisation of society, it also turns all children into potential targets.
A mobilização de crianças­soldados ameaça não só conduzir à criminalização da sociedade, como, além disso, transforma cada criança num alvo.
-------
An assurance was also given that Chairman Kim Chung-il wishes to see a second summit, but that the time must be right for this.
Foi-nos igualmente garantido que o presidente Kim Jong Il encara favoravelmente a realização de uma segunda cimeira, mas que é necessário ag

In [9]:
corpus_en = europarl_en
corpus_en = re.sub(r"\.(?=[0-9]|[a-z]|[A-Z])", ".$$$", corpus_en)
corpus_en = re.sub(r".\$\$\$", "", corpus_en)
corpus_en = re.sub(r" +", " ", corpus_en)
corpus_en = corpus_en.split("\n")

In [10]:
corpus_pt = europarl_pt
corpus_pt = re.sub(r"\.(?=[0-9]|[a-z]|[A-Z])", ".$$$", corpus_pt)
corpus_pt = re.sub(r".\$\$\$", "", corpus_pt)
corpus_pt = re.sub(r" +", " ", corpus_pt)
corpus_pt = corpus_pt.split("\n")

In [11]:
len(corpus_pt), len(corpus_en)

(1960408, 1960408)

# **3. Tokenização**

## **Convertendo de texto para números**

In [12]:
tokenizer_en = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    corpus_en, target_vocab_size=2**13)

In [13]:
tokenizer_pt = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    corpus_pt, target_vocab_size=2**13)

In [14]:
tokenizer_en.vocab_size, tokenizer_pt.vocab_size

(8191, 8116)

In [15]:
vocab_size_en = tokenizer_en.vocab_size + 2
vocab_size_pt = tokenizer_pt.vocab_size + 2

In [16]:
inputs = [[vocab_size_en - 2] + tokenizer_en.encode(sentence) + [vocab_size_en - 1] for sentence in corpus_en]

In [17]:
for _ in range(5):
  print(inputs[random.randint(0, len(inputs) - 1)])

[8191, 25, 23, 71, 39, 999, 304, 10, 356, 2415, 7525, 143, 2, 32, 1, 154, 211, 2, 20, 39, 1556, 40, 1640, 5704, 13, 4, 3516, 5, 1609, 3763, 7967, 2176, 2122, 8050, 2, 4, 97, 7, 207, 6773, 98, 706, 913, 2, 24, 9, 421, 4, 6095, 642, 108, 3680, 2, 5, 1528, 2, 24, 148, 3879, 4, 18, 108, 2415, 6096, 7981, 8192]
[8191, 562, 82, 55, 59, 18, 1323, 2, 11, 33, 14, 2062, 82, 55, 159, 2717, 20, 12, 5, 251, 15, 33, 443, 95, 1, 880, 7981, 8192]
[8191, 62, 8, 304, 11, 300, 22, 1, 236, 86, 7, 293, 658, 3, 334, 121, 9, 2342, 7981, 8192]
[8191, 62, 315, 3, 2862, 2977, 4935, 187, 2, 10, 318, 2, 5, 1076, 2825, 617, 475, 12, 1, 435, 32, 23, 477, 2, 390, 175, 5017, 7967, 3105, 282, 327, 12, 3518, 116, 1072, 10, 1, 154, 126, 6, 1, 23, 113, 7981, 8192]
[8191, 25, 79, 1655, 1419, 12, 4665, 8050, 2, 4321, 7, 1824, 753, 2, 17, 134, 1, 3643, 605, 3, 212, 7981, 8192]


In [18]:
outputs = [[vocab_size_pt - 2] + tokenizer_pt.encode(sentence) + [vocab_size_pt - 1] for sentence in corpus_pt]

In [19]:
for _ in range(5):
  print(outputs[random.randint(0, len(outputs) - 1)])

[8116, 97, 186, 171, 1, 157, 5961, 2053, 39, 252, 2, 1354, 7, 966, 7961, 1, 10, 266, 171, 1, 157, 6280, 4, 59, 9, 305, 7586, 7892, 4479, 5658, 7906, 8117]
[8116, 30, 7856, 48, 7, 40, 67, 408, 2, 1482, 25, 13, 546, 15, 1978, 7, 32, 5, 25, 13, 159, 1425, 5, 1134, 22, 1006, 6495, 8, 188, 7906, 8117]
[8116, 68, 3753, 4, 17, 238, 31, 5664, 19, 1070, 7718, 7892, 136, 16, 591, 2, 1622, 6826, 6689, 7906, 8117]
[8116, 96, 4, 340, 175, 56, 182, 6558, 7975, 1, 1487, 17, 1486, 4484, 6198, 5527, 9, 16, 918, 1320, 29, 4, 3, 32, 71, 7584, 3, 914, 14, 149, 308, 2827, 7975, 1, 838, 7573, 7892, 3860, 2555, 465, 1, 38, 133, 5516, 7021, 465, 1, 22, 503, 10, 4, 2076, 10, 11, 562, 38, 3385, 29, 6, 631, 7, 129, 7906, 8117]
[8116, 36, 759, 4, 197, 3, 1793, 3122, 7892, 13, 674, 523, 7906, 8117]


## **Remoção de sentenças muito longas**

In [20]:
max_length = 15
idx_to_remove = [count for count, sent in enumerate(inputs) if len(sent) > max_length]

In [21]:
len(idx_to_remove)

1685300

In [22]:
for idx in reversed(idx_to_remove):
  del inputs[idx]
  del outputs[idx]

In [23]:
idx_to_remove = [count for count, sent in enumerate(outputs) if len(sent) > max_length]

In [24]:
for idx in reversed(idx_to_remove):
  del inputs[idx]
  del outputs[idx]

In [25]:
len(inputs), len(outputs)

(208990, 208990)

# **4. Padding e Bacthes**

In [26]:
inputs = tf.keras.preprocessing.sequence.pad_sequences(
    inputs, value=0, padding='post', maxlen=max_length)

outputs = tf.keras.preprocessing.sequence.pad_sequences(
    outputs, value=0, padding='post', maxlen=max_length)

In [27]:
for _ in range(5):
  print(outputs[random.randint(0, len(outputs) - 1)])

[8116 1336    3  274  141  242 7906 8117    0    0    0    0    0    0
    0]
[8116 7945 2525 7971 7905 1125   11  851    3  762    3   14  797 7906
 8117]
[8116  373   13  149  469   10  224   24 6130 2390 1894 7906 8117    0
    0]
[8116  296   18   13  674 2066 6011 7906 8117    0    0    0    0    0
    0]
[8116   87    4 1752 7923 8117    0    0    0    0    0    0    0    0
    0]


In [28]:
batch_size = 64
buffer_size = 20000

dataset = tf.data.Dataset.from_tensor_slices((inputs, outputs))
dataset = dataset.cache()
dataset = dataset.shuffle(buffer_size).batch(batch_size)
dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

In [30]:
class PositionalEncoding(layers.Layer):
  def __init__(self):
    super(PositionalEncoding, self).__init__()

  def get_angles(self, pos, i, d_model):
    angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
    return pos * angle_rates

  def call(self, inputs):
    seq_lenght = inputs.shape.as_list()[-2]
    d_model = inputs.shape.as_list()[-1]
    angles = self.get_angles(np.arange(seq_lenght)[:, np.newaxis],
                            np.arange(d_model)[np.newaxis, :],
                            d_model)

    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    pos_encoding = angles[np.newaxis, ...]

    return inputs + tf.cast(pos_encoding, dtype=tf.float32)

# **5. Mecânismo de atenção**

In [31]:
def scaled_dot_product_attention(q, k, v, mask):
  """Calcule os pesos de atenção.
   q, k, v devem ter dimensões principais correspondentes.
   k, v deve ter a penúltima dimensão correspondente, ou seja: seq_len_k = seq_len_v.
   A máscara tem formatos diferentes dependendo do tipo (preenchimento ou antecipação)
   mas deve ser transmitível para adição.

   Argumentos:
     q: formato da consulta == (..., seq_len_q, profundidade)
     k: formato da chave == (..., seq_len_k, profundidade)
     v: forma do valor == (..., seq_len_v, profundidade_v)
     máscara: tensor flutuante com forma transmitível
           para (..., seq_len_q, seq_len_k). O padrão é Nenhum.

   Retorna:
     saída, atenção_pesos
  """

  matmul_qk = tf.matmul(q, k, transpose_b=True)  # (..., seq_len_q, seq_len_k)

  # scale matmul_qk
  dk = tf.cast(tf.shape(k)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

  # adicione a máscara ao tensor escalonado.
  if mask is not None:
    scaled_attention_logits += (mask * -1e9)

  # softmax é normalizado no último eixo (seq_len_k) para que as pontuações
  # some até 1.
  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)  # (..., seq_len_q, seq_len_k)

  output = tf.matmul(attention_weights, v)  # (..., seq_len_q, depth_v)

  return output, attention_weights

In [32]:
class MultiHeadAttention(layers.Layer):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    self.num_heads = num_heads
    self.d_model = d_model

    assert d_model % self.num_heads == 0

    self.depth = d_model // self.num_heads

    self.wq = tf.keras.layers.Dense(d_model)
    self.wk = tf.keras.layers.Dense(d_model)
    self.wv = tf.keras.layers.Dense(d_model)

    self.dense = tf.keras.layers.Dense(d_model)

  def split_heads(self, inputs, batch_size):
    """Split the last dimension into (num_heads, depth).
    Transpose the result such that the shape is (batch_size, num_heads, seq_len, depth)
    """
    inputs = tf.reshape(inputs, (batch_size, -1, self.num_heads, self.depth))
    return tf.transpose(inputs, perm=[0, 2, 1, 3])

  def call(self, v, k, q, mask):
    batch_size = tf.shape(q)[0]

    q = self.wq(q)  # (batch_size, seq_len, d_model)
    k = self.wk(k)  # (batch_size, seq_len, d_model)
    v = self.wv(v)  # (batch_size, seq_len, d_model)

    q = self.split_heads(q, batch_size)  # (batch_size, num_heads, seq_len_q, depth)
    k = self.split_heads(k, batch_size)  # (batch_size, num_heads, seq_len_k, depth)
    v = self.split_heads(v, batch_size)  # (batch_size, num_heads, seq_len_v, depth)

    # scaled_attention.shape == (batch_size, num_heads, seq_len_q, depth)
    # attention_weights.shape == (batch_size, num_heads, seq_len_q, seq_len_k)
    scaled_attention, attention_weights = scaled_dot_product_attention(
        q, k, v, mask)

    scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)

    concat_attention = tf.reshape(scaled_attention,
                                  (batch_size, -1, self.d_model))  # (batch_size, seq_len_q, d_model)

    output = self.dense(concat_attention)  # (batch_size, seq_len_q, d_model)

    return output, attention_weights

# **6. Encoder**

In [33]:
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      layers.Dense(dff, activation='relu'),  # (batch_size, seq_len, dff)
      layers.Dense(d_model)  # (batch_size, seq_len, d_model)
  ])

In [46]:
def get_angles(pos, i, d_model):
  angle_rates = 1 / np.power(10000, (2 * (i//2)) / np.float32(d_model))
  return pos * angle_rates

In [47]:
def positional_encoding(position, d_model):
  angle_rads = get_angles(np.arange(position)[:, np.newaxis],
                          np.arange(d_model)[np.newaxis, :],
                          d_model)

  # apply sin to even indices in the array; 2i
  angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

  # apply cos to odd indices in the array; 2i+1
  angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

  pos_encoding = angle_rads[np.newaxis, ...]

  return tf.cast(pos_encoding, dtype=tf.float32)

In [42]:
class EncoderLayer(layers.Layer):
  def __init__(self, d_model, num_heads, dff, rate=0.1):
    super(EncoderLayer, self).__init__()

    self.mha = MultiHeadAttention(d_model, num_heads)
    self.ffn = point_wise_feed_forward_network(d_model, dff)

    self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = layers.Dropout(rate)
    self.dropout2 = layers.Dropout(rate)

  def call(self, inputs, training, mask):

    attn_output, _ = self.mha(inputs, inputs, inputs, mask)  # (batch_size, input_seq_len, d_model)
    attn_output = self.dropout1(attn_output, training=training)
    out1 = self.layernorm1(inputs + attn_output)  # (batch_size, input_seq_len, d_model)

    ffn_output = self.ffn(out1)  # (batch_size, input_seq_len, d_model)
    ffn_output = self.dropout2(ffn_output, training=training)
    out2 = self.layernorm2(out1 + ffn_output)  # (batch_size, input_seq_len, d_model)

    return out2

In [36]:
class Encoder(layers.Layer):
  def __init__(self, num_layers, d_model, num_heads, dff, input_vocab_size,
               maximum_position_encoding, rate=0.1):
    super(Encoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = layers.Embedding(input_vocab_size, d_model)
    self.pos_encoding = positional_encoding(maximum_position_encoding,
                                            self.d_model)

    self.enc_layers = [EncoderLayer(d_model, num_heads, dff, rate)
                       for _ in range(num_layers)]

    self.dropout = layers.Dropout(rate)

  def call(self, x, training, mask):

    seq_len = tf.shape(x)[1]

    # adicionando embedding e codificação de posições
    x = self.embedding(x)  # (batch_size, input_seq_len, d_model)
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x += self.pos_encoding[:, :seq_len, :]

    x = self.dropout(x, training=training)

    for i in range(self.num_layers):
      x = self.enc_layers[i](x, training, mask)

    return x  # (batch_size, input_seq_len, d_model)

# **7. Decoder**

In [37]:
class DecoderLayer(layers.Layer):
  def __init__(self, d_model, num_heads, dff, rate=0.1):
    super(DecoderLayer, self).__init__()

    self.mha1 = MultiHeadAttention(d_model, num_heads) # camada de atenção 1
    self.mha2 = MultiHeadAttention(d_model, num_heads) # camada de atenção 2

    self.ffn = point_wise_feed_forward_network(d_model, dff)

    self.layernorm1 = layers.LayerNormalization(epsilon=1e-6) # camada de  normalização 1
    self.layernorm2 = layers.LayerNormalization(epsilon=1e-6) # camada de  normalização 2
    self.layernorm3 = layers.LayerNormalization(epsilon=1e-6) # camada de  normalização 3

    self.dropout1 = layers.Dropout(rate) # dropout(0.1)
    self.dropout2 = layers.Dropout(rate) # dropout(0.1)
    self.dropout3 = layers.Dropout(rate) # dropout(0.1)

  def call(self, x, enc_output, training,
           look_ahead_mask, padding_mask):
    # enc_output.shape == (batch_size, input_seq_len, d_model)

    attn1, attn_weights_block1 = self.mha1(x, x, x, look_ahead_mask)  # (batch_size, target_seq_len, d_model)
    attn1 = self.dropout1(attn1, training=training)
    out1 = self.layernorm1(attn1 + x)

    attn2, attn_weights_block2 = self.mha2(
        enc_output, enc_output, out1, padding_mask)  # (batch_size, target_seq_len, d_model)
    attn2 = self.dropout2(attn2, training=training)
    out2 = self.layernorm2(attn2 + out1)  # (batch_size, target_seq_len, d_model)

    ffn_output = self.ffn(out2)  # (batch_size, target_seq_len, d_model)
    ffn_output = self.dropout3(ffn_output, training=training)
    out3 = self.layernorm3(ffn_output + out2)  # (batch_size, target_seq_len, d_model)

    return out3, attn_weights_block1, attn_weights_block2

In [38]:
class Decoder(layers.Layer):
  def __init__(self, num_layers, d_model, num_heads, dff, target_vocab_size,
               maximum_position_encoding, rate=0.1):
    super(Decoder, self).__init__()

    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = layers.Embedding(target_vocab_size, d_model)
    self.pos_encoding = positional_encoding(maximum_position_encoding, d_model)

    self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate)
                       for _ in range(num_layers)]
    self.dropout = layers.Dropout(rate)

  def call(self, x, enc_output, training,
           look_ahead_mask, padding_mask):

    seq_len = tf.shape(x)[1]
    attention_weights = {}

    x = self.embedding(x)  # (batch_size, target_seq_len, d_model)
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x += self.pos_encoding[:, :seq_len, :]

    x = self.dropout(x, training=training)

    for i in range(self.num_layers):
      x, block1, block2 = self.dec_layers[i](x, enc_output, training,
                                             look_ahead_mask, padding_mask)

      attention_weights[f'decoder_layer{i+1}_block1'] = block1
      attention_weights[f'decoder_layer{i+1}_block2'] = block2

    # x.shape == (batch_size, target_seq_len, d_model)
    return x, attention_weights

# **8. Criando o transformer**

In [39]:
def create_padding_mask(seq): # (batch_size, seq_lenght)
  seq = tf.cast(tf.math.equal(seq, 0), tf.float32)

  # add extra dimensions to add the padding
  # to the attention logits.
  return seq[:, tf.newaxis, tf.newaxis, :]  # (batch_size, 1, 1, seq_len)

In [40]:
def create_look_ahead_mask(size):
  mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
  return mask  # (seq_len, seq_len)

In [50]:
class Transformer(tf.keras.Model):
  def __init__(self, num_layers, d_model, num_heads, dff, vocab_size_enc,
               vocab_size_dec, pe_input, pe_target, rate=0.1):
    super().__init__()
    self.encoder = Encoder(num_layers, d_model, num_heads, dff,
                             vocab_size_enc, pe_input, rate)

    self.decoder = Decoder(num_layers, d_model, num_heads, dff,
                           vocab_size_dec, pe_target, rate)

    self.final_layer = tf.keras.layers.Dense(vocab_size_dec)

  def call(self, inputs, training):
    # Keras models prefer if you pass all your inputs in the first argument
    inp, tar = inputs

    enc_padding_mask, look_ahead_mask, dec_padding_mask = self.create_masks(inp, tar)

    enc_output = self.encoder(inp, training, enc_padding_mask)  # (batch_size, inp_seq_len, d_model)

    # dec_output.shape == (batch_size, tar_seq_len, d_model)
    dec_output, attention_weights = self.decoder(
        tar, enc_output, training, look_ahead_mask, dec_padding_mask)

    final_output = self.final_layer(dec_output)  # (batch_size, tar_seq_len, target_vocab_size)

    return final_output, attention_weights

  def create_masks(self, inp, tar):
    # Encoder padding mask
    enc_padding_mask = create_padding_mask(inp)

    # Use no segundo blovo de atenção do decoder
    # Este mascara de preenchimento é usada nas saídas do codificador
    dec_padding_mask = create_padding_mask(inp)

    # Used in the 1st attention block in the decoder.
    # It is used to pad and mask future tokens in the input received by
    # the decoder.
    look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
    dec_target_padding_mask = create_padding_mask(tar)
    look_ahead_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

    return enc_padding_mask, look_ahead_mask, dec_padding_mask

# **9. Definindo hiperparametros**

In [43]:
num_layers = 4
d_model = 128
dff = 512
num_heads = 8
dropout_rate = 0.1

In [53]:
transformer = Transformer(num_layers=num_layers, d_model=d_model,
                          num_heads=num_heads, dff=dff,
                          vocab_size_enc=vocab_size_en, vocab_size_dec=vocab_size_pt,
                          pe_input=1000, pe_target=1000, rate=dropout_rate)

In [54]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

In [55]:
def loss_function(real, pred):
  mask = tf.math.logical_not(tf.math.equal(real, 0))
  loss_ = loss_object(real, pred)

  mask = tf.cast(mask, dtype=loss_.dtype)
  loss_ *= mask

  return tf.reduce_sum(loss_)/tf.reduce_sum(mask)


def accuracy_function(real, pred):
  accuracies = tf.equal(real, tf.argmax(pred, axis=2))

  mask = tf.math.logical_not(tf.math.equal(real, 0))
  accuracies = tf.math.logical_and(mask, accuracies)

  accuracies = tf.cast(accuracies, dtype=tf.float32)
  mask = tf.cast(mask, dtype=tf.float32)
  return tf.reduce_sum(accuracies)/tf.reduce_sum(mask)

In [56]:
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.Mean(name='train_accuracy')

## **Optimizador**

In [85]:
class CustomSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=4000):
        super(CustomSchedule, self).__init__()

        self.d_model = d_model
        self.d_model = tf.cast(self.d_model, tf.float32)

        self.warmup_steps = tf.cast(warmup_steps, tf.float32)

    def __call__(self, step):
        # Cast `step` to float32 for consistency
        step = tf.cast(step, tf.float32)

        # Calculate the learning rate schedule
        arg1 = tf.math.rsqrt(step)
        arg2 = step * tf.math.rsqrt(self.warmup_steps)

        return tf.math.rsqrt(self.d_model) * tf.math.minimum(arg1, arg2)

In [86]:
learning_rate = CustomSchedule(d_model)

optimizer = tf.keras.optimizers.Adam(learning_rate, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

In [88]:
checkpoint_path = "/content/drive/MyDrive/tradutor"

ckpt = tf.train.Checkpoint(transformer=transformer,
                           optimizer=optimizer)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

# if a checkpoint exists, restore the latest checkpoint.
if ckpt_manager.latest_checkpoint:
  ckpt.restore(ckpt_manager.latest_checkpoint)
  print('Latest checkpoint restored!!')

# **10. Treinamento**

In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    start = time.time()

    train_loss.reset_states()
    train_accuracy.reset_states()

    # inp -> portugues, tar -> english
    for (batch, (inp, tar)) in enumerate(dataset):
        dec_inputs = tar[:, :-1]
        dec_outputs = tar[:, 1:]
        with tf.GradientTape() as tape:
            predictions, _ = transformer((inp, dec_inputs), True)
            loss = loss_function(dec_outputs, predictions)

        gradients = tape.gradient(loss, transformer.trainable_variables)
        optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))

        train_loss(loss)
        train_accuracy(dec_outputs, tf.argmax(predictions, axis=-1))


        if batch % 50 == 0:
            print(f'Epoch {epoch + 1} Batch {batch} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')

    if (epoch + 1) % 5 == 0:
        ckpt_save_path = ckpt_manager.save()
        print(f'Saving checkpoint for epoch {epoch+1} at {ckpt_save_path}')

    print(f'Epoch {epoch + 1} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f}')

    print(f'Time taken for 1 epoch: {time.time() - start:.2f} secs\n')